# Extraction Analysis

Recovery (recall/precision vs. ground truth) and hallucination rate for a single extraction run.

In [1]:
import sys
from pathlib import Path
%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

In [8]:
# ── Config ────────────────────────────────────────────────────────────────────
DATASET         = 'pond'
MODEL           = 'llama-3.1-8b'
EXTRACTION_DATE = '2026_04_19'  # set to None to use most recent run

In [9]:
import pandas as pd
from analysis.loaders import load_extraction, load_combined_judgements, load_ground_truth
from scholarlm.utils.unit_conversion import apply_unit_conversion
from analysis.metrics import recovery_rate, hallucination_rate, per_paper_metrics
from analysis.plots import recovery_bar, probability_distribution
from experiments.run_extraction import load_dataset_config

config = load_dataset_config(DATASET)

## Load data

In [10]:
records = load_extraction(DATASET, MODEL, EXTRACTION_DATE)
extraction_df = pd.DataFrame(records)
extraction_df = apply_unit_conversion(extraction_df, config.unit_conversion_table)
print(f'{len(extraction_df)} extracted measurements')
extraction_df.head()

22324 extracted measurements


,title,author,year,paper_code,document_id,name,abbreviations,location,ecosystem,entity_id,...,value,units,page_number,table_number,row_index,column_index,source,context,measurement_id,converted_value
0,analysis of biological indicators related to t...,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,2014,analysis_of_biological,1,Lake Isac,Isac,"centre of the Danube Delta, 45°08′N and 29°17′E",lake,doc_1_entity_0,...,130,mg/l,[1],[None],[None],[None],[text],[and functioning of surface aquatic ecosystems...,0,130.00
1,analysis of biological indicators related to t...,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,2014,analysis_of_biological,1,Lake Isac,Isac,"centre of the Danube Delta, 45°08′N and 29°17′E",lake,doc_1_entity_0,...,130,mg/l,[1],[None],[None],[None],[text],[and functioning of surface aquatic ecosystems...,1,130.00
2,analysis of biological indicators related to t...,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,2014,analysis_of_biological,1,Lake Isac,Isac,"centre of the Danube Delta, 45°08′N and 29°17′E",lake,doc_1_entity_0,...,"Not a numerical value, but a string: '130 mg/l...",mg/l,[1],[None],[None],[None],[text],[and functioning of surface aquatic ecosystems...,2,NaN
3,analysis of biological indicators related to t...,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,2014,analysis_of_biological,1,Merhei,Merhei,"north part of the delta, 45°19′N and 29°26′E",lake,doc_1_entity_2,...,66.33,mg/l,[1],[None],[None],[None],[text],[and functioning of surface aquatic ecosystems...,3,66.33
4,analysis of biological indicators related to t...,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,2014,analysis_of_biological,1,Lake Furtuna,Furtuna,"west of the Danube Delta, 45°21′N – 29°12′E",lake,doc_1_entity_3,...,7.79,NaN,[1],[None],[None],[None],[text],[and functioning of surface aquatic ecosystems...,4,7.79


In [11]:
ground_truth_df = load_ground_truth(config)
print(f'{len(ground_truth_df)} ground truth measurements')
ground_truth_df.head()

3410 ground truth measurements


,author,title,name,location,ecosystem,date,state,attribute,value
0,chopyk et al.,agricultural freshwater pond supports diverse ...,NaN,central maryland; usa,ponds,NaN,NaN,max_depth,3.35
1,chopyk et al.,agricultural freshwater pond supports diverse ...,NaN,central maryland; usa,ponds,NaN,NaN,ph,7.78
2,chopyk et al.,agricultural freshwater pond supports diverse ...,NaN,central maryland; usa,ponds,NaN,NaN,surface_area,2600.00
3,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,analysis of biological indicators related to t...,cuibul cu lebede lake,danube delta,shallow lake,NaN,NaN,ph,8.03
4,tudor; m.; tudor; i-m; ibram; o.; teodorof; l....,analysis of biological indicators related to t...,isac lake,danube delta,shallow lake,NaN,NaN,ph,7.75


In [ ]:
judgements = load_combined_judgements(DATASET, MODEL, EXTRACTION_DATE)
judged_df = pd.DataFrame(judgements)
print(f'{len(judged_df)} judged measurements')

## Recovery

In [12]:
# Update strict_matching to match your dataset's entity key columns
if 'pond' in DATASET:
    STRICT_MATCHING = {'title': 'title', 'attribute': 'attribute', 'converted_value': 'value'}
    FUZZY_MATCHING  = {'name': 'name', 'location': 'location', 'ecosystem': 'ecosystem'}
elif 'nfix' in DATASET:
    extraction_df['attribute'] = extraction_df['attribute'].map({'nfix_rate_areal': 'nfix_rate', 'nfix_rate_volumetric': 'nfix_rate', 'nfix_rate_mass': 'nfix_rate'})
    STRICT_MATCHING = {'paper_code': 'reference_id', 'attribute': 'attribute', 'converted_value': 'value'}
    FUZZY_MATCHING  = {"name": "site_name", "site_type": "habitat"}

stats = recovery_rate(
    extraction_df,
    ground_truth_df,
    strict_matching=STRICT_MATCHING,
    fuzzy_matching=FUZZY_MATCHING,
    cache_path=Path(f"../data/experiments/{DATASET}/extraction/{MODEL}/{EXTRACTION_DATE}/match_cache.pkl")
)
print(stats)

{'recovery': 0.4026392961876833, 'precision': 0.061503314818132954, 'n_extracted': 22324, 'n_gt': 3410}


## Hallucination rate

In [ ]:
hall = hallucination_rate(judged_df)
print(hall)

fig = probability_distribution(judged_df, prob_col='judgement_p_true', label_col='judgement_combined')
fig.savefig('figures/prob_distribution.pdf', bbox_inches='tight')
fig.show()

## Per-paper breakdown

In [ ]:
paper_df = per_paper_metrics(
    extraction_df,
    ground_truth_df,
    judged_df=judged_df,
    strict_matching=STRICT_MATCHING,
    fuzzy_matching=FUZZY_MATCHING,
)
paper_df.sort_values('recall').round(3)